In [ ]:
!pip install --no-index --find-links=/kaggle/input/torch-geometric /kaggle/input/torch-geometric/tg/torch_geometric-2.7.0-py3-none-any.whl
!rm -rf /kaggle/working/*

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch_geometric.data import Data
import torch_geometric.transforms as TG
from tqdm import tqdm
from itertools import permutations, product
import sys
import copy
import math


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Utilities: Random Rotation & Octahedral Group

The octahedral group $O \subset SO(3)$ has 24 elements — the rotational symmetries of a cube.
Each element is a signed permutation matrix with $\det = +1$.

| Type | Count | Description |
|------|-------|-------------|
| Identity | 1 | $I$ |
| Face rotations | 6 | $90°, 270°$ around each coordinate axis |
| Face rotations | 3 | $180°$ around each coordinate axis |
| Vertex rotations | 8 | $120°, 240°$ around 4 body diagonals |
| Edge rotations | 6 | $180°$ around 6 face diagonals |
| **Total** | **24** | |

In [ ]:
def random_rotation_matrix(batch_size=1):
    """Sample uniformly from SO(3) via unit quaternions."""
    u1, u2, u3 = torch.rand(3, batch_size)
    q0 = torch.sqrt(1 - u1) * torch.sin(2 * np.pi * u2)
    q1 = torch.sqrt(1 - u1) * torch.cos(2 * np.pi * u2)
    q2 = torch.sqrt(u1) * torch.sin(2 * np.pi * u3)
    q3 = torch.sqrt(u1) * torch.cos(2 * np.pi * u3)
    R = torch.zeros(batch_size, 3, 3)
    R[:, 0, 0] = 1 - 2 * (q2**2 + q3**2)
    R[:, 0, 1] = 2 * (q1 * q2 - q0 * q3)
    R[:, 0, 2] = 2 * (q1 * q3 + q0 * q2)
    R[:, 1, 0] = 2 * (q1 * q2 + q0 * q3)
    R[:, 1, 1] = 1 - 2 * (q1**2 + q3**2)
    R[:, 1, 2] = 2 * (q2 * q3 - q0 * q1)
    R[:, 2, 0] = 2 * (q1 * q3 - q0 * q2)
    R[:, 2, 1] = 2 * (q2 * q3 + q0 * q1)
    R[:, 2, 2] = 1 - 2 * (q1**2 + q2**2)
    return R


def generate_octahedral_group():
    """Generate all 24 rotation matrices of the octahedral group O ⊂ SO(3).
    
    These are the signed permutation matrices with determinant +1,
    i.e., all ways to permute and optionally negate coordinate axes
    while preserving right-handedness.
    
    Returns:
        torch.Tensor: [24, 3, 3] rotation matrices
    """
    matrices = []
    for perm in permutations([0, 1, 2]):
        for signs in product([-1, 1], repeat=3):
            R = torch.zeros(3, 3)
            for i in range(3):
                R[i, perm[i]] = float(signs[i])
            if torch.det(R) > 0:
                matrices.append(R)
    O_group = torch.stack(matrices)  # [24, 3, 3]
    assert O_group.shape[0] == 24, f"Expected 24 elements, got {O_group.shape[0]}"
    return O_group


# Pre-compute the octahedral group (used throughout training & evaluation)
OCTAHEDRAL_GROUP = generate_octahedral_group()  # [24, 3, 3]
print(f"Octahedral group: {OCTAHEDRAL_GROUP.shape[0]} elements")
print(f"All determinants +1: {torch.allclose(torch.det(OCTAHEDRAL_GROUP), torch.ones(24))}")
print(f"All orthogonal: {torch.allclose(OCTAHEDRAL_GROUP @ OCTAHEDRAL_GROUP.transpose(-1,-2), torch.eye(3).unsqueeze(0).expand(24,-1,-1), atol=1e-6)}")

## Data Loading & Processing

Same ShapeNet pipeline as CHOIR: read `.obj` meshes → sample 2048 surface points → save as `.pt`.

The dataset applies a random $R \sim U(SO(3))$ and returns $(PR, \text{label}, R)$.
For the EQR quotient loss we need the ground-truth rotation $R$.

In [ ]:
download_path = "/kaggle/input/shapenet/02691156"
processed_dir = "output_dir"
MAX_SAMPLES = None 


def custom_read_obj(file_path):
    pos = []
    faces = []
    
    with open(file_path, 'r') as f:
        for line in f:
            components = line.strip().split()
            if not components:
                continue
            if components[0] == 'v':
                pos.append([float(x) for x in components[1:4]])
            elif components[0] == 'f':
                indices = [int(c.split('/')[0]) - 1 for c in components[1:]]
                for i in range(1, len(indices) - 1):
                    faces.append([indices[0], indices[i], indices[i + 1]])

    if not pos:
        return Data(pos=torch.empty((0, 3)), face=torch.empty((2, 0)))

    pos_tensor = torch.tensor(pos, dtype=torch.float)
    
    if faces:
        face_tensor = torch.tensor(faces, dtype=torch.long).t().contiguous()
    else:
        face_tensor = torch.empty((3, 0), dtype=torch.long)
    
    return Data(pos=pos_tensor, face=face_tensor)


def process_shapenet_mesh(data_folder, output_folder=None, num_points=2048, max_samples=float('inf')):
    sampler = TG.SamplePoints(num=num_points, include_normals=False)
    processed_count = 0
    errors = []
    ret = []
    
    print(f"Scanning {data_folder} for models (max {max_samples} samples)...")
    
    for root, _, files in os.walk(data_folder):
        if processed_count >= max_samples:
            break
        for file in files:
            if processed_count >= max_samples:
                break
            if file == "model_normalized.obj":
                file_path = os.path.join(root, file)
                path_parts = os.path.normpath(file_path).split(os.sep)
                
                try:
                    if 'models' in path_parts:
                        models_index = path_parts.index('models')
                        model_id = path_parts[models_index - 1]
                        category_id = path_parts[models_index - 2]
                    else:
                        continue
                    
                    if output_folder:
                        check_save_cat_dir = os.path.join(output_folder, category_id)
                        check_output_path = os.path.join(check_save_cat_dir, f"{model_id}.pt")
                        if os.path.exists(check_output_path):
                            continue

                    data = custom_read_obj(file_path)
                    if data.pos.size(0) == 0:
                        continue

                    data = sampler(data)
                    
                    if output_folder:
                        save_cat_dir = os.path.join(output_folder, category_id)
                        os.makedirs(save_cat_dir, exist_ok=True)
                        output_path = os.path.join(save_cat_dir, f"{model_id}.pt")
                        torch.save(data.pos, output_path)

                    ret.append(data)
                    processed_count += 1
                    if processed_count % 100 == 0:
                        print(f"Processed {processed_count} meshes...")
                        
                except Exception as e:
                    errors.append((file_path, str(e)))
                    
    print(f"Finished. Total processed: {processed_count}")
    if errors:
        print(f"Encountered {len(errors)} errors.")
    return ret


class ShapeNetPointCloudDataset(Dataset):
    def __init__(self, data_dir, num_points=2048, max_samples=None):
        self.data_dir = data_dir
        self.num_points = num_points
        self.samples = []
        self.cat_to_idx = {}
        
        categories = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        for idx, cat in enumerate(categories):
            self.cat_to_idx[cat] = idx
            cat_path = os.path.join(data_dir, cat)
            for file in os.listdir(cat_path):
                if file.endswith(".pt"):
                    self.samples.append((os.path.join(cat_path, file), cat))
                    if max_samples and len(self.samples) >= max_samples:
                        break
            if max_samples and len(self.samples) >= max_samples:
                break

    def __len__(self):
        return len(self.samples)

    def _sample_points(self, pc):
        n = pc.size(0)
        if n == self.num_points:
            return pc
        if n > self.num_points:
            idx = torch.randperm(n)[:self.num_points]
            return pc[idx]
        pad_idx = torch.randint(0, n, (self.num_points - n,))
        return torch.cat([pc, pc[pad_idx]], dim=0)

    def __getitem__(self, idx):
        file_path, cat = self.samples[idx]
        pc = torch.load(file_path, weights_only=True)
        pc = self._sample_points(pc)
        pc = pc - pc.mean(dim=0, keepdim=True)
        label = self.cat_to_idx[cat]
        R = random_rotation_matrix(1)[0]
        pc_rotated = pc @ R
        return pc_rotated, label, R


if not os.path.exists(processed_dir) or not any(
    file.endswith(".pt")
    for _, _, files in os.walk(processed_dir)
    for file in files
):
    process_shapenet_mesh(download_path, output_folder=processed_dir, num_points=2048, max_samples=MAX_SAMPLES or float('inf'))

print(f"Processed dataset ready at: {processed_dir}")

## VNN Layers (Deng et al., ICCV 2021)

SO(3)-equivariant layers operating on **vector features** $[B, C, 3, N]$.
Only the 3-dim rotates; channel dim $C$ and point dim $N$ are untouched.

| Layer | Equivariance guarantee |
|-------|------------------------|
| `VNLinear` | $W$ acts only on $C$ → $f(xR) = f(x)R$ |
| `VNLeakyReLU` | Projection + gating in the 3-dim preserves equivariance |
| `VNBatchNorm` | Rescales by $\|\cdot\|$ (rotation-invariant scalar) |
| `VNMaxPool` | Argmax of dot-products is rotation-invariant |

In [ ]:
EPS = 1e-6

class VNLinear(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(VNLinear, self).__init__()
        self.map_to_feat = nn.Linear(in_channels, out_channels, bias=False)
    
    def forward(self, x):
        '''
        x: point features of shape [B, N_feat, 3, N_samples, ...]
        '''
        x_out = self.map_to_feat(x.transpose(1,-1)).transpose(1,-1)
        return x_out


class VNLeakyReLU(nn.Module):
    def __init__(self, in_channels, share_nonlinearity=False, negative_slope=0.2):
        super(VNLeakyReLU, self).__init__()
        if share_nonlinearity == True:
            self.map_to_dir = nn.Linear(in_channels, 1, bias=False)
        else:
            self.map_to_dir = nn.Linear(in_channels, in_channels, bias=False)
        self.negative_slope = negative_slope
    
    def forward(self, x):
        '''
        x: point features of shape [B, N_feat, 3, N_samples, ...]
        '''
        d = self.map_to_dir(x.transpose(1,-1)).transpose(1,-1)
        dotprod = (x*d).sum(2, keepdim=True)
        mask = (dotprod >= 0).float()
        d_norm_sq = (d*d).sum(2, keepdim=True)
        x_out = self.negative_slope * x + (1-self.negative_slope) * (mask*x + (1-mask)*(x-(dotprod/(d_norm_sq+EPS))*d))
        return x_out


class VNBatchNorm(nn.Module):
    def __init__(self, num_features, dim):
        super(VNBatchNorm, self).__init__()
        self.dim = dim
        if dim == 3 or dim == 4:
            self.bn = nn.BatchNorm1d(num_features)
        elif dim == 5:
            self.bn = nn.BatchNorm2d(num_features)
    
    def forward(self, x):
        '''
        x: point features of shape [B, N_feat, 3, N_samples, ...]
        '''
        norm = torch.norm(x, dim=2) + EPS
        norm_bn = self.bn(norm)
        norm = norm.unsqueeze(2)
        norm_bn = norm_bn.unsqueeze(2)
        x = x / norm * norm_bn
        return x


class VNLinearLeakyReLU(nn.Module):
    def __init__(self, in_channels, out_channels, dim=5, share_nonlinearity=False, negative_slope=0.2):
        super(VNLinearLeakyReLU, self).__init__()
        self.dim = dim
        self.negative_slope = negative_slope
        
        self.map_to_feat = nn.Linear(in_channels, out_channels, bias=False)
        self.batchnorm = VNBatchNorm(out_channels, dim=dim)
        
        if share_nonlinearity == True:
            self.map_to_dir = nn.Linear(in_channels, 1, bias=False)
        else:
            self.map_to_dir = nn.Linear(in_channels, out_channels, bias=False)
    
    def forward(self, x):
        '''
        x: point features of shape [B, N_feat, 3, N_samples, ...]
        '''
        # Linear
        p = self.map_to_feat(x.transpose(1,-1)).transpose(1,-1)
        # BatchNorm
        # p = self.batchnorm(p)
        # LeakyReLU
        d = self.map_to_dir(x.transpose(1,-1)).transpose(1,-1)
        dotprod = (p*d).sum(2, keepdims=True)
        mask = (dotprod >= 0).float()
        d_norm_sq = (d*d).sum(2, keepdims=True)
        x_out = self.negative_slope * p + (1-self.negative_slope) * (mask*p + (1-mask)*(p-(dotprod/(d_norm_sq+EPS))*d))
        return x_out
    


class VNMaxPool(nn.Module):
    def __init__(self, in_channels):
        super(VNMaxPool, self).__init__()
        self.map_to_dir = nn.Linear(in_channels, in_channels, bias=False)
    
    def forward(self, x):
        '''
        x: point features of shape [B, N_feat, 3, N_samples, ...]
        '''
        d = self.map_to_dir(x.transpose(1,-1)).transpose(1,-1)
        dotprod = (x*d).sum(2, keepdims=True)
        idx = dotprod.max(dim=-1, keepdim=False)[1]
        index_tuple = torch.meshgrid([torch.arange(j) for j in x.size()[:-1]]) + (idx,)
        x_max = x[index_tuple]
        return x_max


def mean_pool(x, dim=-1, keepdim=False):
    return x.mean(dim=dim, keepdim=keepdim)

## Stage 1: Equivariant Quotient Orienter $h_\theta$

**Architecture** (Section 4.2 of CHOIR, now with quotient training):

$$h_\theta(PR) = h_\theta(P) \cdot R \quad \forall R \in SO(3) \tag{Eq. 1}$$

- **VNEncoder** $\phi$: maps $\mathbb{R}^{N \times 3} \to \mathbb{R}^{C \times 3 \times N}$ (equivariant features)
- **VNInvariantLayer**: produces equivariant frame $V \in \mathbb{R}^{3 \times 3 \times N}$
- **RotationPredictor** $\psi$: reduces $C \to 2$ channels → Gram-Schmidt → $SO(3)$

Training loss (Scarvelis et al., Eq. 3, on equivariant backbone):
$$\mathcal{L}_{\text{quotient}} = \min_{Q \in O} \| h_\theta(PR) - R Q \|_F^2 \tag{Eq. 2}$$

In [ ]:
class VNEncoder(nn.Module):
    """SO(3)-equivariant encoder φ: R^{N×3} → R^{C×3×N}
    Uses VNN layers (Deng et al., 2021) with [B, C, 3, N] tensor layout.
    φ(PR) = φ(P)R (equivariant by construction).
    """
    def __init__(self, in_channels=1, hidden_channels=64, out_channels=128):
        super().__init__()
        self.conv1 = VNLinearLeakyReLU(in_channels, hidden_channels, dim=4)
        self.conv2 = VNLinearLeakyReLU(hidden_channels, hidden_channels, dim=4)
        self.conv3 = VNLinearLeakyReLU(hidden_channels, out_channels, dim=4)
    
    def forward(self, x):
        # x: [B, N, 3] → [B, 1, 3, N]
        x = x.permute(0, 2, 1).unsqueeze(1)  # [B, 1, 3, N]
        x = self.conv1(x)   # [B, hidden, 3, N]
        x = self.conv2(x)   # [B, hidden, 3, N]
        x = self.conv3(x)   # [B, out, 3, N]
        return x  # [B, C, 3, N]


class VNInvariantLayer(nn.Module):
    """Produces SO(3)-equivariant frame V: R^{C×3×N} → R^{3×3×N}
    Used to compute invariant features via φ(PR)(VR)^T = φ(P)V^T (Eq. 4).
    """
    def __init__(self, in_channels):
        super().__init__()
        self.vn1 = VNLinearLeakyReLU(in_channels, in_channels // 2, dim=4)
        self.vn2 = VNLinearLeakyReLU(in_channels // 2, in_channels // 4, dim=4)
        self.vn_lin = VNLinear(in_channels // 4, 3)
    
    def forward(self, x):
        # x: [B, C, 3, N]
        x = self.vn1(x)      # [B, C//2, 3, N]
        x = self.vn2(x)      # [B, C//4, 3, N]
        V = self.vn_lin(x)   # [B, 3, 3, N]
        return V


class RotationPredictor(nn.Module):
    """SO(3)-equivariant rotation predictor ψ: R^{C×3×N} → SO(3)
    Reduces channels C → 2, mean-pools over N, Gram-Schmidt → SO(3).
    ψ(φ(PR)) = ψ(φ(P))R  (equivariant)
    """
    def __init__(self, in_channels):
        super().__init__()
        self.vn_lin = VNLinear(in_channels, 2)

    def forward(self, x):
        # x: [B, C, 3, N]
        basis = self.vn_lin(x)          # [B, 2, 3, N]
        basis = basis.mean(dim=-1)      # [B, 2, 3]
        v1 = basis[:, 0, :]             # [B, 3]
        v2 = basis[:, 1, :]             # [B, 3]

        # Gram-Schmidt orthonormalisation → SO(3)
        e1 = F.normalize(v1, dim=-1)
        dot = (v2 * e1).sum(dim=-1, keepdim=True)
        e2_proj = v2 - dot * e1
        e2 = F.normalize(e2_proj, dim=-1, eps=1e-6)
        e3 = torch.cross(e1, e2, dim=-1)

        # Stack as columns: R_mat(PR) = R_mat(P) @ R
        R = torch.stack([e1, e2, e3], dim=1)  # [B, 3, 3]
        return R


class OrientationHypothesizer(nn.Module):
    """SO(3)-equivariant orientation hypothesizer h = ψ ∘ φ
    h(PR) = h(P) · R   (Eq. 1)
    Also returns equivariant features φ(PR) and frame V for the residual predictor.
    """
    def __init__(self, hidden_channels=64, out_channels=128):
        super().__init__()
        self.encoder = VNEncoder(1, hidden_channels, out_channels)
        self.rotation_predictor = RotationPredictor(out_channels)
        self.inv_layer = VNInvariantLayer(out_channels)
    
    def forward(self, x):
        # x: [B, N, 3]
        equiv_feat = self.encoder(x)              # [B, C, 3, N]
        R_hyp = self.rotation_predictor(equiv_feat)  # [B, 3, 3]
        V = self.inv_layer(equiv_feat)             # [B, 3, 3, N]
        return R_hyp, equiv_feat, V

## Stage 2: Continuous Invariant Residual Predictor $g_\phi$

Takes **rotation-invariant** inputs:
1. Canonicalized PC: $\tilde{P} = PR \cdot h(PR)^{-1} = P h(P)^{-1}$ (Eq. 3, invariant by equivariance)
2. Invariant features: $\phi(PR)(VR)^T = \phi(P)V^T$ (Eq. 4)

Outputs a **continuous** $\Delta R \in SO(3)$ via 6D representation + Gram-Schmidt.

Unlike Scarvelis et al.'s discrete 24-class flipper, this can express **any** rotation correction.

**Point Transformer** (Zhao et al., 2021) backbone:
$$y_i = \sum_{x_j \in \mathcal{X}(i)} \rho(\gamma(\varphi(x_i) - \psi(x_j) + \delta)) \odot (\alpha(x_j) + \delta)$$

In [ ]:
class InvariantFeatureExtractor(nn.Module):
    """Extracts SO(3)-invariant features: φ(PR)(VR)^T = φ(P)V^T.
    Based on VNN invariant layer principle (Deng et al., 2021, Eq. 13).
    """
    def __init__(self, in_channels, frame_channels=3):
        super().__init__()
        self.in_channels = in_channels
        self.frame_channels = frame_channels
    
    def forward(self, phi_feat, V_feat):
        # phi_feat: [B, C, 3, N] - SO(3)-equivariant feature
        # V_feat:   [B, 3, 3, N] - SO(3)-equivariant frame
        inv_feat = torch.einsum('bcin,bdin->bcdn', phi_feat, V_feat)  # [B, C, 3, N]
        B, C, D, N = inv_feat.shape
        return inv_feat.permute(0, 3, 1, 2).reshape(B, N, C * D)


class PointTransformerLayer(nn.Module):
    """Point Transformer layer (Zhao et al., 2021).
    yi = Σ_{xj ∈ X(i)} ρ(γ(φ(xi) - ψ(xj) + δ)) ⊙ (α(xj) + δ)
    """
    def __init__(self, in_dim, out_dim=None, num_neighbors=16):
        super().__init__()
        out_dim = out_dim or in_dim
        self.num_neighbors = num_neighbors
        self.phi = nn.Linear(in_dim, out_dim)
        self.psi = nn.Linear(in_dim, out_dim)
        self.alpha = nn.Linear(in_dim, out_dim)
        self.pos_enc = nn.Sequential(
            nn.Linear(3, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )
        self.gamma = nn.Sequential(
            nn.Linear(out_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )
        self.out_dim = out_dim
    
    def forward(self, x, pos):
        # x: [B, N, D], pos: [B, N, 3]
        B, N, D = x.shape
        k = min(self.num_neighbors, N)
        
        with torch.no_grad():
            dists = torch.cdist(pos, pos)
            _, knn_idx = dists.topk(k, dim=-1, largest=False)
        
        batch_idx = torch.arange(B, device=x.device).view(B, 1, 1).expand(B, N, k)
        
        x_neighbors = x[batch_idx, knn_idx]
        pos_neighbors = pos[batch_idx, knn_idx]
        
        phi_x = self.phi(x).unsqueeze(2)
        psi_x = self.psi(x_neighbors)
        alpha_x = self.alpha(x_neighbors)
        
        pos_diff = pos.unsqueeze(2) - pos_neighbors
        delta = self.pos_enc(pos_diff)
        
        attn_input = phi_x - psi_x + delta
        attn_weights = self.gamma(attn_input)
        attn_weights = F.softmax(attn_weights, dim=2)
        
        out = (attn_weights * (alpha_x + delta)).sum(dim=2)
        return out


class PointTransformerBlock(nn.Module):
    """Residual Point Transformer block: linear → PT → linear + residual."""
    def __init__(self, dim, num_neighbors=16):
        super().__init__()
        self.linear_in = nn.Linear(dim, dim)
        self.pt_layer = PointTransformerLayer(dim, dim, num_neighbors)
        self.linear_out = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
    
    def forward(self, x, pos):
        identity = x
        x = self.linear_in(x)
        x = self.pt_layer(x, pos)
        x = self.linear_out(x)
        x = x + identity
        return x


class ResidualPredictor(nn.Module):
    """SO(3)-invariant continuous residual predictor g (Section 4.3 of EQR).
    
    Takes rotation-invariant inputs:
      canon_pc: [B, N, 3]      hypothetically canonicalized PC (invariant)
      phi_feat: [B, C, 3, N]   equivariant features from encoder
      V_feat:   [B, 3, 3, N]   equivariant frame from invariant layer
    
    Outputs ΔR ∈ SO(3) via 6D representation + Gram-Schmidt.
    Unlike Scarvelis et al.'s 24-class flipper, this can express any rotation.
    """
    def __init__(self, in_channels=128, hidden_dim=256, inv_feat_channels=3, num_neighbors=16):
        super().__init__()
        self.inv_feat_extractor = InvariantFeatureExtractor(in_channels, inv_feat_channels)
        inv_feat_dim = in_channels * inv_feat_channels
        
        self.point_embed = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        self.feat_embed = nn.Sequential(
            nn.Linear(inv_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        self.pt_block1 = PointTransformerBlock(hidden_dim, num_neighbors)
        self.pt_block2 = PointTransformerBlock(hidden_dim, num_neighbors)
        
        # 6D rotation representation (Zhou et al.)
        self.out_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 6)
        )
    
    def forward(self, canon_pc, phi_feat, V_feat):
        inv_feat = self.inv_feat_extractor(phi_feat, V_feat)  # [B, N, C*3]
        
        point_emb = self.point_embed(canon_pc)
        feat_emb = self.feat_embed(inv_feat)
        
        combined = point_emb + feat_emb
        
        combined = self.pt_block1(combined, canon_pc)
        combined = self.pt_block2(combined, canon_pc)
        
        pooled = combined.mean(dim=1)  # [B, hidden]
        out = self.out_mlp(pooled)     # [B, 6]
        
        # 6D → SO(3) via Gram-Schmidt
        v1 = out[:, :3]
        v2 = out[:, 3:]
        e1 = F.normalize(v1, dim=-1)
        e2_proj = v2 - (v2 * e1).sum(dim=-1, keepdim=True) * e1
        e2 = F.normalize(e2_proj, dim=-1, eps=1e-6)
        e3 = torch.cross(e1, e2, dim=-1)
        R_residual = torch.stack([e1, e2, e3], dim=1)  # [B, 3, 3]
        return R_residual

## EQR Model: Equivariant Quotient-Residual Pipeline

**Full pipeline (Figure 1)**:
$$f(PR) = g\bigl(\phi_{\text{inv}}(\tilde{P})\bigr) \cdot h_\theta(PR) \tag{Eq. 6}$$

where $\tilde{P} = PR \cdot h(PR)^{-1}$ is the rotation-invariant canonicalized PC.

**Equivariance proof (Proposition 2)**:
- $g(\cdot)$ receives invariant inputs → outputs same $\Delta R$ regardless of input rotation
- $h(PR) = h(P)R$ by VNN equivariance
- $f(PR) = \Delta R \cdot h(P) \cdot R = f(P) \cdot R$ ✓

In [ ]:
class EQR(nn.Module):
    """Equivariant Quotient-Residual Pipeline.
    
    Combines:
    1. SO(3)-equivariant quotient orienter h (VNN backbone, quotient loss by O)
       → predicts orientation up to octahedral equivalence
    2. SO(3)-invariant continuous residual predictor g (Point Transformer backbone)
       → refines with continuous ΔR ∈ SO(3), replacing discrete 24-class flipper
    
    Final prediction: f(PR) = g(z) · h(PR)
    Equivariance: f(PR) = f(P) · R   (exact, by construction)
    
    Key advantages over prior work:
    - vs CHOIR: quotient loss handles symmetric shapes (avoids mean-collapse)
    - vs Scarvelis: equivariant backbone gives exact stability (no augmentation needed)
    - vs Scarvelis: continuous residual instead of discrete 24-class flipper
    """
    def __init__(self, hidden_channels=64, out_channels=128, residual_hidden_dim=256, num_neighbors=16):
        super().__init__()
        self.hypothesizer = OrientationHypothesizer(hidden_channels, out_channels)
        self.residual_predictor = ResidualPredictor(
            out_channels, 
            residual_hidden_dim, 
            inv_feat_channels=3,
            num_neighbors=num_neighbors
        )
    
    def forward(self, x):
        # x: [B, N, 3] input point cloud (possibly rotated: PR)
        R_hyp, equiv_feat, V_feat = self.hypothesizer(x)
        # R_hyp:      [B, 3, 3] - h(PR), equivariant orientation hypothesis
        # equiv_feat: [B, C, 3, N] - φ(PR), equivariant features
        # V_feat:     [B, 3, 3, N] - VR, equivariant frame
        
        # Eq. 3: Canonicalized PC (invariant)
        # P̃ = PR · h(PR)^{-1} = P · h(P)^{-1}  (invariant to input rotation)
        canon_pc = x @ R_hyp.transpose(-1, -2)    # [B, N, 3]
        
        # Residual prediction from invariant inputs
        # g receives: canon_pc (invariant), equiv_feat & V_feat (for invariant features via Eq. 4)
        R_res = self.residual_predictor(canon_pc, equiv_feat, V_feat)  # [B, 3, 3] ΔR
        
        # Eq. 6: f(PR) = ΔR · h(PR)
        R_final = R_res @ R_hyp                    # [B, 3, 3]
        return R_final, R_hyp, R_res, equiv_feat
    
    def forward_hypothesis_only(self, x):
        """Run only the equivariant quotient orienter (Stage 1)."""
        R_hyp, equiv_feat, V_feat = self.hypothesizer(x)
        return R_hyp, equiv_feat, V_feat
    
    def canonicalize(self, x):
        """Cancel characteristic orientation: PR f(PR)^T = P f(P)^T (invariant)."""
        R_final, _, _, _ = self.forward(x)
        return x @ R_final.transpose(-1, -2)  # [B, N, 3]

## Loss Functions & Evaluation Metrics

**Stage 1 — Octahedral quotient loss** (Scarvelis et al., Eq. 3):
$$\mathcal{L}_{\text{quotient}} = \min_{Q \in O} \|h(PR) - RQ\|_F^2$$

**Stage 2 — Geodesic residual loss** (Geist et al., ICML 2024):
$$\mathcal{L}_{\text{residual}} = d_{\text{geo}}(\Delta R,\, (Q^*)^{-1})^2 = \arccos\!\left(\tfrac{\mathrm{tr}(\Delta R^\top (Q^*)^{-1}) - 1}{2}\right)^2$$

where $Q^* = \arg\min_{Q \in O} \|h(PR) - RQ\|_F^2$

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  LOSS FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def quotient_loss(R_pred, R_gt, O_group):
    """Octahedral quotient regression loss (Scarvelis et al., Eq. 3).
    
    L = min_{Q ∈ O} ||R_pred - R_gt @ Q||²_F
    
    Quotienting by O avoids mean-collapse for shapes with symmetry ⊆ O.
    
    Args:
        R_pred:  [B, 3, 3]  predicted rotation (h(PR))
        R_gt:    [B, 3, 3]  ground truth rotation R
        O_group: [24, 3, 3] octahedral group elements
    Returns:
        scalar loss
    """
    B = R_pred.shape[0]
    # R_gt @ Q for all Q ∈ O: [B, 1, 3, 3] @ [1, 24, 3, 3] → [B, 24, 3, 3]
    R_gt_Q = R_gt.unsqueeze(1) @ O_group.unsqueeze(0)  # [B, 24, 3, 3]
    diff = R_pred.unsqueeze(1) - R_gt_Q                # [B, 24, 3, 3]
    frob_sq = (diff ** 2).sum(dim=(-2, -1))            # [B, 24]
    min_frob_sq = frob_sq.min(dim=1)[0]                # [B]
    return min_frob_sq.mean()


def find_closest_octahedral(R_pred, R_gt, O_group):
    """Find Q* = argmin_{Q ∈ O} ||R_pred - R_gt @ Q||²_F.
    
    Used to determine the residual target (Q*)^{-1} for Stage 2 training.
    
    Args:
        R_pred:  [B, 3, 3]  predicted rotation h(PR)
        R_gt:    [B, 3, 3]  ground truth rotation R
        O_group: [24, 3, 3] octahedral group elements
    Returns:
        Q_star: [B, 3, 3]  closest octahedral element
        idx:    [B]        index into O_group
    """
    B = R_pred.shape[0]
    R_gt_Q = R_gt.unsqueeze(1) @ O_group.unsqueeze(0)  # [B, 24, 3, 3]
    diff = R_pred.unsqueeze(1) - R_gt_Q
    frob_sq = (diff ** 2).sum(dim=(-2, -1))            # [B, 24]
    idx = frob_sq.argmin(dim=1)                        # [B]
    Q_star = O_group[idx]                              # [B, 3, 3]
    return Q_star, idx


def geodesic_loss(R_pred, R_target):
    """Geodesic distance squared on SO(3).
    
    d_geo(R1, R2)² = arccos((tr(R1^T R2) - 1) / 2)²
    
    Continuous and smooth, unlike cross-entropy on 24 classes.
    
    Args:
        R_pred:   [B, 3, 3]
        R_target: [B, 3, 3]
    Returns:
        scalar loss
    """
    R_diff = R_pred.transpose(-1, -2) @ R_target
    trace = R_diff.diagonal(dim1=-2, dim2=-1).sum(-1)  # [B]
    trace = torch.clamp(trace, -1.0 + 1e-7, 3.0 - 1e-7)
    angle = torch.acos((trace - 1.0) / 2.0)           # [B]
    return (angle ** 2).mean()


def residual_loss(R_res, R_hyp, R_gt, O_group):
    """Stage 2 loss: train residual predictor g to predict (Q*)^{-1}.
    
    Given h(PR) ≈ R Q*, the target for g is (Q*)^T so that
    f(PR) = g(z) · h(PR) ≈ (Q*)^T · R Q* → recovers canonical orientation.
    
    Uses geodesic loss for continuous, smooth optimization on SO(3)
    (vs. Scarvelis et al.'s discrete 24-class cross-entropy).
    
    Args:
        R_res:   [B, 3, 3]  predicted residual ΔR from g
        R_hyp:   [B, 3, 3]  hypothesis h(PR) from Stage 1 (frozen)
        R_gt:    [B, 3, 3]  ground truth rotation R
        O_group: [24, 3, 3] octahedral group
    Returns:
        scalar loss (geodesic distance²)
    """
    with torch.no_grad():
        Q_star, _ = find_closest_octahedral(R_hyp, R_gt, O_group)
        target = Q_star.transpose(-1, -2)  # (Q*)^{-1} = (Q*)^T
    return geodesic_loss(R_res, target)


def full_pipeline_loss(R_final, R_gt, O_group):
    """End-to-end quotient loss on the full pipeline output.
    
    L = min_{Q ∈ O} ||f(PR) - R Q||²_F
    
    Used in Phase 3 (joint fine-tuning) to refine both stages together.
    """
    return quotient_loss(R_final, R_gt, O_group)


# ═══════════════════════════════════════════════════════════════════════════
#  EVALUATION METRICS
# ═══════════════════════════════════════════════════════════════════════════

def rotation_angle(R1, R2):
    """Geodesic angle between two rotations in radians."""
    R_diff = R1.transpose(-1, -2) @ R2
    trace = R_diff.diagonal(dim1=-2, dim2=-1).sum(-1)
    trace = torch.clamp(trace, -1.0 + 1e-7, 3.0 - 1e-7)
    angle = torch.acos((trace - 1.0) / 2.0)
    return angle


def chordal_l2_mean(rotations):
    """Chordal L2 mean (Karcher/Fréchet mean on SO(3) via SVD projection)."""
    Ce = rotations.sum(dim=0)
    U, _, Vt = torch.linalg.svd(Ce)
    det = torch.det(U @ Vt)
    if det < 0:
        U[:, -1] *= -1
    S = U @ Vt
    return S


def compute_stability(model, pc, num_rotations=10):
    """Stability metric (CHOIR Eq. 1).
    Same shape, different rotations → how consistent is f(PR)R^T?
    Lower is better (0° = perfect stability, guaranteed by equivariance).
    """
    model.eval()
    device_m = next(model.parameters()).device
    pc = pc.to(device_m)

    with torch.no_grad():
        rotations = random_rotation_matrix(num_rotations).to(device_m)
        net_rotations = []

        for i in range(num_rotations):
            R = rotations[i]
            pc_rot = pc @ R
            R_pred, _, _, _ = model(pc_rot.unsqueeze(0))
            R_pred = R_pred[0]
            R_net = R @ R_pred.transpose(-1, -2)
            net_rotations.append(R_net)

        net_rotations = torch.stack(net_rotations)
        R_mean = chordal_l2_mean(net_rotations)
        angles = rotation_angle(
            net_rotations,
            R_mean.unsqueeze(0).expand(num_rotations, -1, -1)
        )
        stability = torch.sqrt((angles ** 2).mean()).item() * 180.0 / np.pi

    return stability


def compute_consistency(model, dataset, num_samples=50):
    """Consistency metric (CHOIR Eq. 2).
    Different shapes of same class → how similar are their canonical orientations?
    Lower is better.
    """
    model.eval()
    device_m = next(model.parameters()).device
    orientations = []

    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            file_path, cat = dataset.samples[i]
            pc = torch.load(file_path, weights_only=True)
            pc = dataset._sample_points(pc)
            pc = pc - pc.mean(dim=0, keepdim=True)
            pc = pc.to(device_m)
            R_pred, _, _, _ = model(pc.unsqueeze(0))
            R_pred = R_pred[0].cpu()
            orientations.append(R_pred)

    if len(orientations) < 2:
        return 0.0

    orientations = torch.stack(orientations)
    R_mean = chordal_l2_mean(orientations)
    angles = rotation_angle(
        orientations,
        R_mean.unsqueeze(0).expand(len(orientations), -1, -1)
    )
    consistency = torch.sqrt((angles ** 2).mean()).item() * 180.0 / np.pi
    return consistency


def compute_quotient_error(model, dataset, O_group, num_samples=50):
    """Quotient angular error: min_{Q ∈ O} angle(f(PR), RQ) in degrees.
    Measures how well the full pipeline predicts orientation up to O.
    """
    model.eval()
    device_m = next(model.parameters()).device
    O_dev = O_group.to(device_m)
    errors = []

    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            pc_rot, label, R_gt = dataset[i]
            pc_rot = pc_rot.unsqueeze(0).to(device_m)
            R_gt = R_gt.unsqueeze(0).to(device_m)
            R_pred, _, _, _ = model(pc_rot)
            
            # min_{Q ∈ O} angle(R_pred, R_gt @ Q)
            R_gt_Q = R_gt @ O_dev.unsqueeze(0)  # [1, 24, 3, 3]
            angles = rotation_angle(
                R_pred.unsqueeze(1).expand(-1, 24, -1, -1).reshape(-1, 3, 3),
                R_gt_Q.reshape(-1, 3, 3)
            ).reshape(1, 24)  # [1, 24]
            min_angle = angles.min(dim=1)[0]  # [1]
            errors.append(min_angle.item() * 180.0 / np.pi)

    return np.mean(errors), np.std(errors)

## 3-Phase Training Protocol

| Phase | What trains | Loss | Purpose |
|-------|-----------|------|---------|
| **1** Warm-up | $h_\theta$ only | $\mathcal{L}_{\text{quotient}}$ | Establish reliable equivariant baseline |
| **2** Residual | $g_\phi$ only ($h_\theta$ frozen) | $\mathcal{L}_{\text{residual}}$ | Learn continuous correction |
| **3** Joint | $h_\theta + g_\phi$ | $\mathcal{L}_{\text{quotient}} + \lambda \mathcal{L}_{\text{residual}}$ | Fine-tune jointly |

In [ ]:
CHECKPOINT_INPUT_DIR = "/kaggle/input/eqr-checkpoint"   # upload previous checkpoints here
CHECKPOINT_SAVE_DIR  = "/kaggle/working"                 # writable output dir


def save_checkpoint(model, optimizer, scheduler, epoch, loss, phase, is_best=False):
    state = {
        'epoch': epoch,
        'phase': phase,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss,
    }
    current_path = os.path.join(CHECKPOINT_SAVE_DIR, f'eqr_phase{phase}_current.pt')
    torch.save(state, current_path)
    if is_best:
        best_path = os.path.join(CHECKPOINT_SAVE_DIR, f'eqr_phase{phase}_best.pt')
        torch.save(state, best_path)
        print(f"  [Checkpoint] Phase {phase} best saved at epoch {epoch}, loss={loss:.6f}")


def load_checkpoint(model, optimizer=None, scheduler=None, phase=1):
    """Try loading checkpoint for the given phase."""
    for fname in [f'eqr_phase{phase}_current.pt', f'eqr_phase{phase}_best.pt']:
        path = os.path.join(CHECKPOINT_INPUT_DIR, fname)
        if os.path.exists(path):
            print(f"  [Checkpoint] Loading from {path}")
            state = torch.load(path, map_location=device, weights_only=True)
            model.load_state_dict(state['model_state_dict'])
            if optimizer is not None:
                optimizer.load_state_dict(state['optimizer_state_dict'])
            if scheduler is not None:
                scheduler.load_state_dict(state['scheduler_state_dict'])
            start_epoch = state['epoch'] + 1
            best_loss   = state['loss']
            print(f"  [Checkpoint] Phase {phase} resuming from epoch {start_epoch}, best_loss={best_loss:.6f}")
            return start_epoch, best_loss
    print(f"  [Checkpoint] No Phase {phase} checkpoint found, starting from scratch.")
    return 0, float('inf')


def load_best_checkpoint(model, phase):
    """Load best checkpoint for a phase (for transitioning between phases)."""
    for directory in [CHECKPOINT_SAVE_DIR, CHECKPOINT_INPUT_DIR]:
        path = os.path.join(directory, f'eqr_phase{phase}_best.pt')
        if os.path.exists(path):
            state = torch.load(path, map_location=device, weights_only=True)
            model.load_state_dict(state['model_state_dict'])
            print(f"  [Checkpoint] Loaded Phase {phase} best from {path} (loss={state['loss']:.6f})")
            return True
    return False


# ═══════════════════════════════════════════════════════════════════════════
#  PHASE 1: Train quotient orienter h_θ
# ═══════════════════════════════════════════════════════════════════════════

def train_phase1(model, train_loader, optimizer, scheduler, O_group, epochs=200):
    """Phase 1: Train SO(3)-equivariant quotient orienter.
    
    Loss: min_{Q ∈ O} ||h(PR) - RQ||²_F  (Eq. 2)
    Only h_θ parameters are optimized.
    """
    O_dev = O_group.to(device)
    
    # Freeze residual predictor during Phase 1
    for param in model.residual_predictor.parameters():
        param.requires_grad = False
    for param in model.hypothesizer.parameters():
        param.requires_grad = True
    
    start_epoch, best_loss = load_checkpoint(model, optimizer, scheduler, phase=1)
    
    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0
        for pc_rot, labels, R_gt in train_loader:
            pc_rot = pc_rot.to(device)
            R_gt = R_gt.to(device)
            
            optimizer.zero_grad()
            R_hyp, _, _ = model.forward_hypothesis_only(pc_rot)  # [B, 3, 3]
            loss = quotient_loss(R_hyp, R_gt, O_dev)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        
        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        
        is_best = avg_loss < best_loss
        if is_best:
            best_loss = avg_loss
        save_checkpoint(model, optimizer, scheduler, epoch, best_loss, phase=1, is_best=is_best)
        
        if (epoch + 1) % 10 == 0 or is_best:
            print(f"  [Phase 1] Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}, "
                  f"Best: {best_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    
    # Restore best
    load_best_checkpoint(model, phase=1)
    print(f"  [Phase 1] Complete. Best loss: {best_loss:.6f}")


# ═══════════════════════════════════════════════════════════════════════════
#  PHASE 2: Train residual predictor g_φ (h frozen)
# ═══════════════════════════════════════════════════════════════════════════

def train_phase2(model, train_loader, optimizer, scheduler, O_group, epochs=200):
    """Phase 2: Train SO(3)-invariant continuous residual predictor.
    
    h_θ is FROZEN. Only g_φ is trained.
    Loss: d_geo(ΔR, (Q*)^{-1})²  where Q* = argmin_Q ||h(PR) - RQ||²  (Eq. 7)
    """
    O_dev = O_group.to(device)
    
    # Freeze hypothesizer, unfreeze residual predictor
    for param in model.hypothesizer.parameters():
        param.requires_grad = False
    for param in model.residual_predictor.parameters():
        param.requires_grad = True
    
    start_epoch, best_loss = load_checkpoint(model, optimizer, scheduler, phase=2)
    
    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0
        for pc_rot, labels, R_gt in train_loader:
            pc_rot = pc_rot.to(device)
            R_gt = R_gt.to(device)
            
            optimizer.zero_grad()
            R_final, R_hyp, R_res, _ = model(pc_rot)
            loss = residual_loss(R_res, R_hyp.detach(), R_gt, O_dev)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        
        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        
        is_best = avg_loss < best_loss
        if is_best:
            best_loss = avg_loss
        save_checkpoint(model, optimizer, scheduler, epoch, best_loss, phase=2, is_best=is_best)
        
        if (epoch + 1) % 10 == 0 or is_best:
            print(f"  [Phase 2] Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}, "
                  f"Best: {best_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    
    load_best_checkpoint(model, phase=2)
    print(f"  [Phase 2] Complete. Best loss: {best_loss:.6f}")


# ═══════════════════════════════════════════════════════════════════════════
#  PHASE 3: Joint fine-tuning
# ═══════════════════════════════════════════════════════════════════════════

def train_phase3(model, train_loader, optimizer, scheduler, O_group, epochs=100, lambda_res=1.0):
    """Phase 3: Joint fine-tuning of h_θ and g_φ.
    
    Loss: L_quotient(h) + λ · L_residual(g)  + L_full(f)
    All parameters are trainable.
    """
    O_dev = O_group.to(device)
    
    # Unfreeze everything
    for param in model.parameters():
        param.requires_grad = True
    
    start_epoch, best_loss = load_checkpoint(model, optimizer, scheduler, phase=3)
    
    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0
        for pc_rot, labels, R_gt in train_loader:
            pc_rot = pc_rot.to(device)
            R_gt = R_gt.to(device)
            
            optimizer.zero_grad()
            R_final, R_hyp, R_res, _ = model(pc_rot)
            
            # Stage 1 loss: quotient on hypothesis
            l_quot = quotient_loss(R_hyp, R_gt, O_dev)
            # Stage 2 loss: geodesic on residual
            l_res = residual_loss(R_res, R_hyp, R_gt, O_dev)
            # End-to-end quotient loss on full prediction
            l_full = full_pipeline_loss(R_final, R_gt, O_dev)
            
            loss = l_quot + lambda_res * l_res + l_full
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
        
        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        
        is_best = avg_loss < best_loss
        if is_best:
            best_loss = avg_loss
        save_checkpoint(model, optimizer, scheduler, epoch, best_loss, phase=3, is_best=is_best)
        
        if (epoch + 1) % 10 == 0 or is_best:
            print(f"  [Phase 3] Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}, "
                  f"Best: {best_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")
    
    load_best_checkpoint(model, phase=3)
    print(f"  [Phase 3] Complete. Best loss: {best_loss:.6f}")

In [ ]:
print("=" * 60)
print("TRAINING EQR MODEL (Equivariant Quotient-Residual)")
print("=" * 60)

# ── Hyperparameters ────────────────────────────────────────────────────────
NUM_POINTS = 1024
BATCH_SIZE = 16

# Phase durations
PHASE1_EPOCHS = 200    # Quotient orienter warm-up
PHASE2_EPOCHS = 200    # Residual predictor (h frozen)
PHASE3_EPOCHS = 100    # Joint fine-tuning

PHASE1_LR = 0.001
PHASE2_LR = 0.001
PHASE3_LR = 0.0001     # Lower LR for fine-tuning

LAMBDA_RES = 1.0        # Weight of residual loss in Phase 3

# ── Dataset ─────────────────────────────────────────────────────────────────
base_dataset = ShapeNetPointCloudDataset(processed_dir, num_points=NUM_POINTS, max_samples=None)
train_loader = DataLoader(base_dataset, batch_size=BATCH_SIZE, shuffle=True)

# ── Model ──────────────────────────────────────────────────────────────────
model = EQR(
    hidden_channels=64,
    out_channels=64,
    residual_hidden_dim=128,
    num_neighbors=16
).to(device)

print(f"Dataset size: {len(base_dataset)}")
print(f"Number of classes: {len(base_dataset.cat_to_idx)}")
print(f"Points per cloud: {NUM_POINTS}")
print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Hypothesizer:       {sum(p.numel() for p in model.hypothesizer.parameters()):,}")
print(f"  Residual predictor: {sum(p.numel() for p in model.residual_predictor.parameters()):,}")
print()

# ── Phase 1: Quotient orienter ─────────────────────────────────────────────
print("─" * 60)
print("PHASE 1: Equivariant Quotient Orienter")
print(f"  Training h_θ with L_quotient = min_{{Q∈O}} ||h(PR) - RQ||²_F")
print(f"  Epochs: {PHASE1_EPOCHS}, LR: {PHASE1_LR}")
print("─" * 60)

optimizer1 = torch.optim.Adam(
    model.hypothesizer.parameters(), lr=PHASE1_LR
)
scheduler1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer1, T_max=PHASE1_EPOCHS)

train_phase1(model, train_loader, optimizer1, scheduler1, OCTAHEDRAL_GROUP, epochs=PHASE1_EPOCHS)

# ── Phase 2: Residual predictor ────────────────────────────────────────────
print()
print("─" * 60)
print("PHASE 2: Continuous Invariant Residual Predictor")
print(f"  h_θ FROZEN. Training g_φ with L_residual = d_geo(ΔR, (Q*)⁻¹)²")
print(f"  Epochs: {PHASE2_EPOCHS}, LR: {PHASE2_LR}")
print("─" * 60)

optimizer2 = torch.optim.Adam(
    model.residual_predictor.parameters(), lr=PHASE2_LR
)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=PHASE2_EPOCHS)

train_phase2(model, train_loader, optimizer2, scheduler2, OCTAHEDRAL_GROUP, epochs=PHASE2_EPOCHS)

# ── Phase 3: Joint fine-tuning ─────────────────────────────────────────────
print()
print("─" * 60)
print("PHASE 3: Joint Fine-Tuning")
print(f"  All parameters trainable. L = L_quot + {LAMBDA_RES}·L_res + L_full")
print(f"  Epochs: {PHASE3_EPOCHS}, LR: {PHASE3_LR}")
print("─" * 60)

optimizer3 = torch.optim.Adam(
    model.parameters(), lr=PHASE3_LR
)
scheduler3 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer3, T_max=PHASE3_EPOCHS)

train_phase3(model, train_loader, optimizer3, scheduler3, OCTAHEDRAL_GROUP, 
             epochs=PHASE3_EPOCHS, lambda_res=LAMBDA_RES)

print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

In [ ]:
import matplotlib.pyplot as plt

# Fetch a batch
batch_size = 12
samples = [base_dataset[i] for i in range(batch_size)]

# Plot
fig, axes = plt.subplots(3, 4, figsize=(12, 12))
axes = axes.flatten()

for idx, (pc, label, R) in enumerate(samples):
    ax = axes[idx]
    pc_np = pc.cpu().numpy()
    ax.scatter(pc_np[:, 0], pc_np[:, 1], s=1, alpha=0.6)
    ax.set_title(f'Sample {idx}')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pointcloud_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## Evaluation

| Metric | What it measures | EQR advantage |
|--------|-----------------|---------------|
| **Stability** | $f(PR_1)R_1^T \approx f(PR_2)R_2^T$ (same shape, diff rotations) | Exact 0° by VNN equivariance |
| **Consistency** | $f(P_1) \approx f(P_2)$ (different shapes, same class) | Quotient loss avoids collapse |
| **Quotient error** | $\min_{Q \in O} \angle(f(PR), RQ)$ | Direct measure of prediction quality |
| **Equivariance** | $f(PR) = f(P)R$ | Exact by construction |

In [ ]:
print("=" * 60)
print("EVALUATION")
print("=" * 60)

model.eval()
test_dataset = ShapeNetPointCloudDataset(processed_dir, num_points=NUM_POINTS, max_samples=None)

# ── Stability ──────────────────────────────────────────────────────────────
print("\nComputing stability (same shape, different rotations)...")
sample_stabilities = []
for i in range(min(10, len(test_dataset))):
    pc, _, _ = test_dataset[i]
    pc = pc.to(device)
    stability = compute_stability(model, pc, num_rotations=10)
    sample_stabilities.append(stability)

print(f"  Stability: {np.mean(sample_stabilities):.2f} +/- {np.std(sample_stabilities):.2f} degrees")
print(f"  (Exact 0° is guaranteed by VNN equivariance — nonzero values are numerical noise)")

# ── Consistency ─────────────────────────────────────────────────────────────
print("\nComputing consistency (different shapes, same class)...")
consistency = compute_consistency(model, test_dataset, num_samples=min(100, len(test_dataset)))
print(f"  Consistency: {consistency:.2f} degrees")

# ── Quotient Error ──────────────────────────────────────────────────────────
print("\nComputing quotient angular error...")
q_mean, q_std = compute_quotient_error(model, test_dataset, OCTAHEDRAL_GROUP, 
                                        num_samples=min(100, len(test_dataset)))
print(f"  Quotient error: {q_mean:.2f} +/- {q_std:.2f} degrees")
print(f"  (min_{{Q∈O}} angle(f(PR), RQ) — measures orientation prediction quality)")

# ── Equivariance Verification ───────────────────────────────────────────────
print("\nTesting equivariance on trained model...")
x = torch.randn(2, NUM_POINTS, 3).to(device)
R_test = random_rotation_matrix(1)[0].to(device)
x_rot = x @ R_test

with torch.no_grad():
    R_final1, R_hyp1, R_res1, _ = model(x)
    R_final2, R_hyp2, R_res2, _ = model(x_rot)
    
    # h(PR) vs h(P)R
    R_hyp1_rot = R_hyp1 @ R_test
    print(f"  h(PR) vs h(P)R error: {(R_hyp2 - R_hyp1_rot).abs().max().item():.6f}")
    
    # g(z_1) vs g(z_2) — should be identical (invariant)
    print(f"  g(z) invariance error: {(R_res2 - R_res1).abs().max().item():.6f}")
    
    # f(PR) vs f(P)R
    R_final1_rot = R_final1 @ R_test
    print(f"  f(PR) vs f(P)R error: {(R_final2 - R_final1_rot).abs().max().item():.6f}")
    
    # Canonical PC consistency
    canon1 = x @ R_final1.transpose(-1, -2)
    canon2 = x_rot @ R_final2.transpose(-1, -2)
    print(f"  Canonical PC difference: {(canon2 - canon1).abs().max().item():.6f}")

print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)

## Detailed Equivariance / Invariance Tests

Tests every layer individually to verify correctness:
- **EQUIVARIANT**: $f(xR) = f(x)R$ — output rotates with input
- **INVARIANT**: $f(xR) = f(x)$ — output unchanged under rotation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  EQUIVARIANCE / INVARIANCE TESTS
# ══════════════════════════════════════════════════════════════════════════

def rot_vn_4d(x, R):
    """Rotate the 3-dim of [B, C, 3, N] tensor by R [3,3]."""
    return (x.permute(0, 1, 3, 2) @ R).permute(0, 1, 3, 2)

def rot_vn_3d(x, R):
    """Rotate the last dim of [B, C, 3] tensor by R [3,3]."""
    return x @ R

def rot_pc(x, R):
    """Standard point cloud rotation [B, N, 3] by R [3,3]."""
    return x @ R

def check(label, got, want, tol=1e-4, tag="EQUIVARIANT"):
    err = (got - want).abs().max().item()
    ok  = "✓" if err < tol else "✗"
    print(f"  {ok} {label:<32s}  max|delta| = {err:.2e}  [{tag}]")

# ── setup ──────────────────────────────────────────────────────────────────
device_t = next(model.parameters()).device
model.eval()

pc, _, _ = base_dataset[1]
pc = pc[:NUM_POINTS].unsqueeze(0).to(device_t)   # [1, N, 3]

R      = random_rotation_matrix(1)[0].to(device_t)
pc_rot = rot_pc(pc, R)

B, N = pc.shape[0], pc.shape[1]

print("=" * 68)
print("  EQUIVARIANCE / INVARIANCE TESTS")
print("=" * 68)

with torch.no_grad():

    # ── 1. VNLinear ────────────────────────────────────────────────────────
    C_in, C_out = 8, 16
    vn_lin = VNLinear(C_in, C_out).to(device_t).eval()
    x_P  = torch.randn(B, C_in, 3, N, device=device_t)
    x_PR = rot_vn_4d(x_P, R)
    out_P  = vn_lin(x_P)
    out_PR = vn_lin(x_PR)
    want   = rot_vn_4d(out_P, R)
    check("VNLinear", out_PR, want)

    # ── 2. VNLeakyReLU ────────────────────────────────────────────────────
    C = 16
    vn_relu = VNLeakyReLU(C).to(device_t).eval()
    x_P  = torch.randn(B, C, 3, N, device=device_t)
    x_PR = rot_vn_4d(x_P, R)
    out_P  = vn_relu(x_P)
    out_PR = vn_relu(x_PR)
    want   = rot_vn_4d(out_P, R)
    check("VNLeakyReLU", out_PR, want)

    # ── 3. VNBatchNorm ────────────────────────────────────────────────────
    C = 16
    vn_bn = VNBatchNorm(C, dim=4).to(device_t)
    vn_bn.train()
    vn_bn(torch.randn(4, C, 3, N, device=device_t))
    vn_bn.eval()
    x_P  = torch.randn(B, C, 3, N, device=device_t)
    x_PR = rot_vn_4d(x_P, R)
    out_P  = vn_bn(x_P)
    out_PR = vn_bn(x_PR)
    want   = rot_vn_4d(out_P, R)
    check("VNBatchNorm", out_PR, want)

    # ── 4. VNLinearLeakyReLU ──────────────────────────────────────────────
    C_in, C_out = 8, 16
    vn_llr = VNLinearLeakyReLU(C_in, C_out, dim=4).to(device_t)
    vn_llr.train()
    vn_llr(torch.randn(4, C_in, 3, N, device=device_t))
    vn_llr.eval()
    x_P  = torch.randn(B, C_in, 3, N, device=device_t)
    x_PR = rot_vn_4d(x_P, R)
    out_P  = vn_llr(x_P)
    out_PR = vn_llr(x_PR)
    want   = rot_vn_4d(out_P, R)
    check("VNLinearLeakyReLU", out_PR, want)

    # ── 5. VNMaxPool ──────────────────────────────────────────────────────
    C = 16
    vn_pool = VNMaxPool(C).to(device_t).eval()
    x_P  = torch.randn(B, C, 3, N, device=device_t)
    x_PR = rot_vn_4d(x_P, R)
    out_P  = vn_pool(x_P)
    out_PR = vn_pool(x_PR)
    want   = rot_vn_3d(out_P, R)
    check("VNMaxPool", out_PR, want)

    # ── 6. VNEncoder ──────────────────────────────────────────────────────
    enc = model.hypothesizer.encoder
    feat_P  = enc(pc)
    feat_PR = enc(pc_rot)
    want    = rot_vn_4d(feat_P, R)
    check("VNEncoder", feat_PR, want)

    # ── 7. VNInvariantLayer ───────────────────────────────────────────────
    inv_layer = model.hypothesizer.inv_layer
    V_P  = inv_layer(feat_P)
    V_PR = inv_layer(feat_PR)
    want = rot_vn_4d(V_P, R)
    check("VNInvariantLayer", V_PR, want)

    # ── 8. RotationPredictor ──────────────────────────────────────────────
    rot_pred = model.hypothesizer.rotation_predictor
    R_hyp_P  = rot_pred(feat_P)
    R_hyp_PR = rot_pred(feat_PR)
    want     = R_hyp_P @ R
    check("RotationPredictor", R_hyp_PR, want)

    # ── 9. OrientationHypothesizer ────────────────────────────────────────
    hyp = model.hypothesizer
    R_h_P,  phi_P,  V_P2  = hyp(pc)
    R_h_PR, phi_PR, V_PR2 = hyp(pc_rot)
    want = R_h_P @ R
    check("OrientationHypothesizer", R_h_PR, want)

    # ── 10. InvariantFeatureExtractor ─────────────────────────────────────
    inv_ext = model.residual_predictor.inv_feat_extractor
    inv_P  = inv_ext(phi_P,  V_P2)
    inv_PR = inv_ext(phi_PR, V_PR2)
    check("InvariantFeatureExtractor", inv_PR, inv_P, tag="INVARIANT")

    # ── 11. ResidualPredictor ─────────────────────────────────────────────
    res_pred = model.residual_predictor
    canon_P  = pc      @ R_h_P.transpose(-1, -2)
    canon_PR = pc_rot  @ R_h_PR.transpose(-1, -2)
    R_res_P  = res_pred(canon_P,  phi_P,  V_P2)
    R_res_PR = res_pred(canon_PR, phi_PR, V_PR2)
    check("ResidualPredictor", R_res_PR, R_res_P, tag="INVARIANT")

    # ── 12. EQR full model ────────────────────────────────────────────────
    R_f_P,  _, _, _ = model(pc)
    R_f_PR, _, _, _ = model(pc_rot)
    want = R_f_P @ R
    check("EQR full model", R_f_PR, want)

    # bonus: canonicalized PC invariance
    canon1 = pc      @ R_f_P.transpose(-1, -2)
    canon2 = pc_rot  @ R_f_PR.transpose(-1, -2)
    check("Canonical PC", canon2, canon1, tag="INVARIANT")

print()
print("  max|delta| < 1e-4  →  property holds numerically")
print("  EQUIVARIANT: f(P·R) == f(P) @ R")
print("  INVARIANT  : f(P·R) == f(P)  (same output)")
print("=" * 68)